In [6]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import mnist
from sklearn.cluster import KMeans
import pickle
def makefile(what,filename):
    with open(filename,"wb") as f3:
        pickle.dump(what,f3)

def readfile(filename):
    with open(filename,"rb") as f4:
        ans=pickle.load(f4)
    return ans
from math import *

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from tensorflow.keras.datasets import fashion_mnist as fmnist
from tensorflow.keras.datasets import cifar10

In [4]:
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

In [1]:
import time


In [20]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[50]
window_size=[2,3,5,7,11,15]
step_size=[2,3,5,7,10]
for e in range(1):
    
    for K in num_cluster:
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        for ss in step_size:
            print("step:",ss)
            for c in range(1):
                kmeans=KMeans(n_clusters=K)
                GMM=GaussianMixture(n_components=1)
                IF=IsolationForest()
                LOF=LocalOutlierFactor(novelty=True)
                OCSVM=OneClassSVM()
                

                print(c)
                train = X_train[y_train==c]
                #result=[]
                full_LCB=np.zeros([len(train),0])
                full_LCB_test=np.zeros([len(X_test),0])
                for size in window_size:
                    print(size)
                    start=time.time()
                    i_size=0
                    i_index=[]
                    for i in range(32-size+1):
                        if i %ss ==0:
                            i_index.append(i)
                            i_size+=1
                            
                    j_size=0
                    j_index=[]
                    for j in range(32-size+1):
                        if j %ss ==0:
                            j_index.append(j)
                            j_size+=1
                    i_index=np.array(i_index)
                    j_index=np.array(j_index)
                    
                    print(size)
                    position=np.array(range(size**2)).reshape(size,size)
                    windows=np.zeros([len(train),i_size,j_size,size*size*3])
                    for i in range(size):
                        for j in range(size):
                            num=position[i][j]
                            windows[:,:,:,num*3:num*3+3]=train[:,i_index+i][:,:,j_index+j]
                    vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
                    kmeans.fit(vector)
                    cluster=kmeans.predict(vector)
                    cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
                    #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
                    LCB=[]
                    for raw in cluster:
                        balance=np.zeros(K)
                        for value in raw:
                            balance[value]+=1
                        LCB.append(balance)
                    LCB=np.array(LCB)
                    LCB=LCB/cluster.shape[1]
                    full_LCB=np.concatenate([full_LCB,LCB],axis=1)
                    end=time.time()
                    tim=end-start
                    print(tim)
                    start=time.time()
                    windows_test=np.zeros([len(X_test),i_size,j_size,size*size*3])
                    for i in range(size):
                        for j in range(size):
                            num=position[i][j]
                            windows_test[:,:,:,num*3:num*3+3]=X_test[:,i_index+i][:,:,j_index+j]
                    vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
                    cluster_test=kmeans.predict(vector_test)
                    cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
                    LCB_test=[]
                    for raw in cluster_test:
                        balance=np.zeros(K)
                        for value in raw:
                            balance[value]+=1
                        LCB_test.append(balance)
                    LCB_test=np.array(LCB_test)
                    LCB_test=LCB_test/cluster_test.shape[1]
                    full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
                    end=time.time()
                    tim=end-start
                    print(tim)
                GMM.fit(full_LCB)
                IF.fit(full_LCB)
                LOF.fit(full_LCB)
                OCSVM.fit(full_LCB)
                LP=full_LCB.mean(axis=0)
                Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
                GMM_seen=GMM.score_samples(full_LCB_test)
                IF_seen=IF.score_samples(full_LCB_test)
                LOF_seen=LOF.score_samples(full_LCB_test)
                OCSVM_seen=OCSVM.score_samples(full_LCB_test)
                y_eval=np.zeros(len(y_test))
                y_eval[y_test==c]=1
                Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
                GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
                IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
                LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
                OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
                #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
                print(Classic_result)
                print(OCSVM_result)
                print(LOF_result)
                print(IF_result)
                print(GMM_result)
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)
        print(np.array(Classic_result).mean())
        print(np.array(OCSVM_result).mean())
        print(np.array(LOF_result).mean())
        print(np.array(IF_result).mean())
        print(np.array(GMM_result).mean())

step: 2
0
2
2
4.8026204109191895
0.8021740913391113
3
3
6.307667255401611
0.9832766056060791
5
5
8.32923173904419
1.5698463916778564
7
7
22.534594774246216
2.3529036045074463
11
11
34.74018859863281
3.7889671325683594
15
15
47.78167176246643
5.0311667919158936
[np.float64(44.3)]
[np.float64(41.9)]
[np.float64(49.2)]
[np.float64(59.6)]
[np.float64(57.6)]
step: 3
0
2
2
1.7952141761779785
0.5249757766723633
3
3
2.2243504524230957
0.48224425315856934
5
5
5.916076183319092
0.8887450695037842
7
7
7.447511434555054
1.217782735824585
11
11
13.134203672409058
2.177746295928955
15
15
18.549733638763428
2.4373645782470703
[np.float64(44.3), np.float64(43.6)]
[np.float64(41.9), np.float64(41.0)]
[np.float64(49.2), np.float64(48.9)]
[np.float64(59.6), np.float64(52.1)]
[np.float64(57.6), np.float64(57.9)]
step: 5
0
2
2
0.6146175861358643
0.2413017749786377
3
3
0.7055118083953857
0.19162845611572266
5
5
1.7408950328826904
0.3728642463684082
7
7
2.9856083393096924
0.6159942150115967
11
11
7.116098880

In [21]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[50]
window_size=[2,3,5,7,11,15]
step_size=[1]
for e in range(1):
    
    for K in num_cluster:
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        for ss in step_size:
            print("step:",ss)
            for c in range(1):
                kmeans=KMeans(n_clusters=K)
                GMM=GaussianMixture(n_components=1)
                IF=IsolationForest()
                LOF=LocalOutlierFactor(novelty=True)
                OCSVM=OneClassSVM()
                

                print(c)
                train = X_train[y_train==c]
                #result=[]
                full_LCB=np.zeros([len(train),0])
                full_LCB_test=np.zeros([len(X_test),0])
                for size in window_size:
                    print(size)
                    start=time.time()
                    i_size=0
                    i_index=[]
                    for i in range(32-size+1):
                        if i %ss ==0:
                            i_index.append(i)
                            i_size+=1
                            
                    j_size=0
                    j_index=[]
                    for j in range(32-size+1):
                        if j %ss ==0:
                            j_index.append(j)
                            j_size+=1
                    i_index=np.array(i_index)
                    j_index=np.array(j_index)
                    
                    print(size)
                    position=np.array(range(size**2)).reshape(size,size)
                    windows=np.zeros([len(train),i_size,j_size,size*size*3])
                    for i in range(size):
                        for j in range(size):
                            num=position[i][j]
                            windows[:,:,:,num*3:num*3+3]=train[:,i_index+i][:,:,j_index+j]
                    vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
                    kmeans.fit(vector)
                    cluster=kmeans.predict(vector)
                    cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
                    #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
                    LCB=[]
                    for raw in cluster:
                        balance=np.zeros(K)
                        for value in raw:
                            balance[value]+=1
                        LCB.append(balance)
                    LCB=np.array(LCB)
                    LCB=LCB/cluster.shape[1]
                    full_LCB=np.concatenate([full_LCB,LCB],axis=1)
                    end=time.time()
                    tim=end-start
                    print(tim)
                    start=time.time()
                    windows_test=np.zeros([len(X_test),i_size,j_size,size*size*3])
                    for i in range(size):
                        for j in range(size):
                            num=position[i][j]
                            windows_test[:,:,:,num*3:num*3+3]=X_test[:,i_index+i][:,:,j_index+j]
                    vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
                    cluster_test=kmeans.predict(vector_test)
                    cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
                    LCB_test=[]
                    for raw in cluster_test:
                        balance=np.zeros(K)
                        for value in raw:
                            balance[value]+=1
                        LCB_test.append(balance)
                    LCB_test=np.array(LCB_test)
                    LCB_test=LCB_test/cluster_test.shape[1]
                    full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
                    end=time.time()
                    tim=end-start
                    print(tim)
                GMM.fit(full_LCB)
                IF.fit(full_LCB)
                LOF.fit(full_LCB)
                OCSVM.fit(full_LCB)
                LP=full_LCB.mean(axis=0)
                Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
                GMM_seen=GMM.score_samples(full_LCB_test)
                IF_seen=IF.score_samples(full_LCB_test)
                LOF_seen=LOF.score_samples(full_LCB_test)
                OCSVM_seen=OCSVM.score_samples(full_LCB_test)
                y_eval=np.zeros(len(y_test))
                y_eval[y_test==c]=1
                Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
                GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
                IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
                LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
                OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
                #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
                print(Classic_result)
                print(OCSVM_result)
                print(LOF_result)
                print(IF_result)
                print(GMM_result)
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)
        print(np.array(Classic_result).mean())
        print(np.array(OCSVM_result).mean())
        print(np.array(LOF_result).mean())
        print(np.array(IF_result).mean())
        print(np.array(GMM_result).mean())

step: 1
0
2
2
18.75964069366455
3.0433566570281982
3
3
23.559255123138428
3.9589571952819824
5
5
44.77294182777405
6.249024152755737
7
7
88.57793712615967
9.298091888427734
11
11
149.2681827545166
14.747974872589111
15
15
186.0704221725464
18.90821623802185
[np.float64(44.9)]
[np.float64(41.8)]
[np.float64(50.1)]
[np.float64(58.1)]
[np.float64(58.0)]
[np.float64(44.9)]
[np.float64(41.8)]
[np.float64(50.1)]
[np.float64(58.1)]
[np.float64(58.0)]
44.9
41.8
50.1
58.1
58.0


In [23]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[2,5,10, 25,50]
window_size=[15]
step_size=[1]
for e in range(1):
    
    for K in num_cluster:
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        for ss in step_size:
            print("step:",ss)
            for c in range(1):
                kmeans=KMeans(n_clusters=K)
                GMM=GaussianMixture(n_components=1)
                IF=IsolationForest()
                LOF=LocalOutlierFactor(novelty=True)
                OCSVM=OneClassSVM()
                

                print(c)
                train = X_train[y_train==c]
                #result=[]
                full_LCB=np.zeros([len(train),0])
                full_LCB_test=np.zeros([len(X_test),0])
                for size in window_size:
                    print(size)
                    start=time.time()
                    i_size=0
                    i_index=[]
                    for i in range(32-size+1):
                        if i %ss ==0:
                            i_index.append(i)
                            i_size+=1
                            
                    j_size=0
                    j_index=[]
                    for j in range(32-size+1):
                        if j %ss ==0:
                            j_index.append(j)
                            j_size+=1
                    i_index=np.array(i_index)
                    j_index=np.array(j_index)
                    
                    print(size)
                    position=np.array(range(size**2)).reshape(size,size)
                    windows=np.zeros([len(train),i_size,j_size,size*size*3])
                    for i in range(size):
                        for j in range(size):
                            num=position[i][j]
                            windows[:,:,:,num*3:num*3+3]=train[:,i_index+i][:,:,j_index+j]
                    vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
                    kmeans.fit(vector)
                    cluster=kmeans.predict(vector)
                    cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
                    #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
                    LCB=[]
                    for raw in cluster:
                        balance=np.zeros(K)
                        for value in raw:
                            balance[value]+=1
                        LCB.append(balance)
                    LCB=np.array(LCB)
                    LCB=LCB/cluster.shape[1]
                    full_LCB=np.concatenate([full_LCB,LCB],axis=1)
                    end=time.time()
                    tim=end-start
                    print(tim)
                    start=time.time()
                    windows_test=np.zeros([len(X_test),i_size,j_size,size*size*3])
                    for i in range(size):
                        for j in range(size):
                            num=position[i][j]
                            windows_test[:,:,:,num*3:num*3+3]=X_test[:,i_index+i][:,:,j_index+j]
                    vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
                    cluster_test=kmeans.predict(vector_test)
                    cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
                    LCB_test=[]
                    for raw in cluster_test:
                        balance=np.zeros(K)
                        for value in raw:
                            balance[value]+=1
                        LCB_test.append(balance)
                    LCB_test=np.array(LCB_test)
                    LCB_test=LCB_test/cluster_test.shape[1]
                    full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
                    end=time.time()
                    tim=end-start
                    print(tim)
                GMM.fit(full_LCB)
                IF.fit(full_LCB)
                LOF.fit(full_LCB)
                OCSVM.fit(full_LCB)
                LP=full_LCB.mean(axis=0)
                Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
                GMM_seen=GMM.score_samples(full_LCB_test)
                IF_seen=IF.score_samples(full_LCB_test)
                LOF_seen=LOF.score_samples(full_LCB_test)
                OCSVM_seen=OCSVM.score_samples(full_LCB_test)
                y_eval=np.zeros(len(y_test))
                y_eval[y_test==c]=1
                Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
                GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
                IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
                LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
                OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
                #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
                print(Classic_result)
                print(OCSVM_result)
                print(LOF_result)
                print(IF_result)
                print(GMM_result)
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)
        print(np.array(Classic_result).mean())
        print(np.array(OCSVM_result).mean())
        print(np.array(LOF_result).mean())
        print(np.array(IF_result).mean())
        print(np.array(GMM_result).mean())

step: 1
0
15
15
25.476430892944336
19.040499448776245
[np.float64(51.0)]
[np.float64(46.0)]
[np.float64(49.1)]
[np.float64(47.3)]
[np.float64(51.0)]
[np.float64(51.0)]
[np.float64(46.0)]
[np.float64(49.1)]
[np.float64(47.3)]
[np.float64(51.0)]
51.0
46.0
49.1
47.3
51.0
step: 1
0
15
15
36.720945596694946
18.972890615463257
[np.float64(53.5)]
[np.float64(51.5)]
[np.float64(53.5)]
[np.float64(55.2)]
[np.float64(52.5)]
[np.float64(53.5)]
[np.float64(51.5)]
[np.float64(53.5)]
[np.float64(55.2)]
[np.float64(52.5)]
53.5
51.5
53.5
55.2
52.5
step: 1
0
15
15
60.95237469673157
18.942956686019897
[np.float64(51.3)]
[np.float64(51.7)]
[np.float64(55.6)]
[np.float64(48.1)]
[np.float64(53.1)]
[np.float64(51.3)]
[np.float64(51.7)]
[np.float64(55.6)]
[np.float64(48.1)]
[np.float64(53.1)]
51.3
51.7
55.6
48.1
53.1
step: 1
0
15
15
156.81431555747986
19.077735424041748
[np.float64(52.0)]
[np.float64(53.8)]
[np.float64(60.9)]
[np.float64(48.2)]
[np.float64(57.7)]
[np.float64(52.0)]
[np.float64(53.8)]
[np.flo

In [10]:
vector_test.shape

(3240000, 675)

In [18]:
pd.DataFrame(np.array(GMM_result).reshape(5,10)).mean(axis=1)

0    67.76
1    66.49
2    65.31
3    63.56
4    62.43
dtype: float64

In [19]:
pd.DataFrame(np.array(GMM_result).reshape(5,10)).to_csv("GMM_CIFAR10_fusion.csv")

In [24]:
%%time
X_train=fmnist.load_data()[0][0]
y_train=fmnist.load_data()[0][1]
X_test=fmnist.load_data()[1][0]
y_test=fmnist.load_data()[1][1]
num_cluster=[50]
window_size=[2,3,5,7,11,15]
step_size=[1,2,3,5,7,10]


for K in num_cluster:
    Classic_result=[]
    GMM_result=[]
    IF_result=[]
    LOF_result=[]
    OCSVM_result=[]
    for ss in step_size:
        print("step:",ss)
        for c in range(1):
            kmeans=KMeans(n_clusters=K)
            GMM=GaussianMixture(n_components=1)
            IF=IsolationForest()
            LOF=LocalOutlierFactor(novelty=True)
            OCSVM=OneClassSVM()
            
            print(c)
            train = X_train[y_train==c]
            #result=[]
            full_LCB=np.zeros([len(train),0])
            full_LCB_test=np.zeros([len(X_test),0])
            for size in window_size:
                print(size)
                start=time.time()
                i_size=0
                i_index=[]
                for i in range(28-size+1):
                    if i %ss ==0:
                        i_index.append(i)
                        i_size+=1
                        
                j_size=0
                j_index=[]
                for j in range(28-size+1):
                    if j %ss ==0:
                        j_index.append(j)
                        j_size+=1
                i_index=np.array(i_index)
                j_index=np.array(j_index)
                
                #print(size)
                position=np.array(range(size**2)).reshape(size,size)
                windows=np.zeros([len(train),i_size,j_size,size*size])
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,:,num]=train[:,i_index+i][:,:,j_index+j]                
                vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
                
                kmeans.fit(vector)
                cluster=kmeans.predict(vector)
                cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
                #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
                LCB=[]
                for raw in cluster:
                    balance=np.zeros(K)
                    for value in raw:
                        balance[value]+=1
                    LCB.append(balance)
                LCB=np.array(LCB)
                LCB=LCB/cluster.shape[1]
                full_LCB=np.concatenate([full_LCB,LCB],axis=1)
                end=time.time()
                tim=end-start
                print(tim)
                start=time.time()
                windows_test=np.zeros([len(X_test),i_size,j_size,size*size])
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows_test[:,:,:,num]=X_test[:,i_index+i][:,:,j_index+j]
                vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
                cluster_test=kmeans.predict(vector_test)
                cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
                LCB_test=[]
                for raw in cluster_test:
                    balance=np.zeros(K)
                    for value in raw:
                        balance[value]+=1
                    LCB_test.append(balance)
                LCB_test=np.array(LCB_test)
                LCB_test=LCB_test/cluster_test.shape[1]
                full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
                end=time.time()
                tim=end-start
                print(tim)
            GMM.fit(full_LCB)
            IF.fit(full_LCB)
            LOF.fit(full_LCB)
            OCSVM.fit(full_LCB)
            LP=full_LCB.mean(axis=0)
            Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(full_LCB_test)
            IF_seen=IF.score_samples(full_LCB_test)
            LOF_seen=LOF.score_samples(full_LCB_test)
            OCSVM_seen=OCSVM.score_samples(full_LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
        #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
    print(Classic_result)
    print(GMM_result)
    print(IF_result)
    print(LOF_result)
    print(OCSVM_result)
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

step: 1
0
2
10.575551509857178
2.4797956943511963
3
13.919528245925903
2.0875391960144043
5
18.69790744781494
2.6565568447113037
7
22.618791580200195
3.578367233276367
11
37.55970764160156
4.971801042556763
15
49.75575137138367
5.385758876800537
step: 2
0
2
2.789792776107788
0.6224453449249268
3
3.613715648651123
0.5480802059173584
5
5.1675190925598145
0.7108545303344727
7
7.51497745513916
0.9673569202423096
11
8.824846744537354
1.328418493270874
15
6.737693548202515
1.4584546089172363
step: 3
0
2
0.9432218074798584
0.25698018074035645
3
1.5688190460205078
0.2846343517303467
5
1.8721895217895508
0.35245203971862793
7
2.8823740482330322
0.5188093185424805
11
2.261657953262329
0.6302077770233154
15
4.3308844566345215
0.7890987396240234
step: 5
0
2
0.3931276798248291
0.1250162124633789
3
0.44947052001953125
0.12745118141174316
5
0.48116302490234375
0.1250298023223877
7
1.343524694442749
0.2184581756591797
11
0.6670393943786621
0.2975425720214844
15
1.0685083866119385
0.30153393745422363
s

In [26]:
%%time
X_train=fmnist.load_data()[0][0]
y_train=fmnist.load_data()[0][1]
X_test=fmnist.load_data()[1][0]
y_test=fmnist.load_data()[1][1]
num_cluster=[2,5,10,25]
window_size=[15]
step_size=[1]


for K in num_cluster:
    Classic_result=[]
    GMM_result=[]
    IF_result=[]
    LOF_result=[]
    OCSVM_result=[]
    for ss in step_size:
        print("step:",ss)
        for c in range(1):
            kmeans=KMeans(n_clusters=K)
            GMM=GaussianMixture(n_components=1)
            IF=IsolationForest()
            LOF=LocalOutlierFactor(novelty=True)
            OCSVM=OneClassSVM()
            
            print(c)
            train = X_train[y_train==c]
            #result=[]
            full_LCB=np.zeros([len(train),0])
            full_LCB_test=np.zeros([len(X_test),0])
            for size in window_size:
                print(size)
                start=time.time()
                i_size=0
                i_index=[]
                for i in range(28-size+1):
                    if i %ss ==0:
                        i_index.append(i)
                        i_size+=1
                        
                j_size=0
                j_index=[]
                for j in range(28-size+1):
                    if j %ss ==0:
                        j_index.append(j)
                        j_size+=1
                i_index=np.array(i_index)
                j_index=np.array(j_index)
                
                #print(size)
                position=np.array(range(size**2)).reshape(size,size)
                windows=np.zeros([len(train),i_size,j_size,size*size])
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,:,num]=train[:,i_index+i][:,:,j_index+j]                
                vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
                
                kmeans.fit(vector)
                cluster=kmeans.predict(vector)
                cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
                #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
                LCB=[]
                for raw in cluster:
                    balance=np.zeros(K)
                    for value in raw:
                        balance[value]+=1
                    LCB.append(balance)
                LCB=np.array(LCB)
                LCB=LCB/cluster.shape[1]
                full_LCB=np.concatenate([full_LCB,LCB],axis=1)
                end=time.time()
                tim=end-start
                print(tim)
                start=time.time()
                windows_test=np.zeros([len(X_test),i_size,j_size,size*size])
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows_test[:,:,:,num]=X_test[:,i_index+i][:,:,j_index+j]
                vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
                cluster_test=kmeans.predict(vector_test)
                cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
                LCB_test=[]
                for raw in cluster_test:
                    balance=np.zeros(K)
                    for value in raw:
                        balance[value]+=1
                    LCB_test.append(balance)
                LCB_test=np.array(LCB_test)
                LCB_test=LCB_test/cluster_test.shape[1]
                full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
                end=time.time()
                tim=end-start
                print(tim)
            GMM.fit(full_LCB)
            IF.fit(full_LCB)
            LOF.fit(full_LCB)
            OCSVM.fit(full_LCB)
            LP=full_LCB.mean(axis=0)
            Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(full_LCB_test)
            IF_seen=IF.score_samples(full_LCB_test)
            LOF_seen=LOF.score_samples(full_LCB_test)
            OCSVM_seen=OCSVM.score_samples(full_LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
        #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
    print(Classic_result)
    print(GMM_result)
    print(IF_result)
    print(LOF_result)
    print(OCSVM_result)
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

step: 1
0
15
6.743093967437744
5.259912014007568
[np.float64(57.5)]
[np.float64(57.5)]
[np.float64(61.1)]
[np.float64(53.2)]
[np.float64(65.3)]
step: 1
0
15
10.13151216506958
5.329509496688843
[np.float64(58.0)]
[np.float64(70.8)]
[np.float64(75.6)]
[np.float64(64.2)]
[np.float64(71.8)]
step: 1
0
15
12.872260332107544
5.115284204483032
[np.float64(60.2)]
[np.float64(79.4)]
[np.float64(76.1)]
[np.float64(72.4)]
[np.float64(75.3)]
step: 1
0
15
17.390279054641724
5.132312059402466
[np.float64(67.5)]
[np.float64(81.9)]
[np.float64(69.3)]
[np.float64(75.6)]
[np.float64(77.1)]
CPU times: user 4min 51s, sys: 6.21 s, total: 4min 57s
Wall time: 1min 15s


In [25]:
%%time
X_train=mnist.load_data()[0][0]
y_train=mnist.load_data()[0][1]
X_test=mnist.load_data()[1][0]
y_test=mnist.load_data()[1][1]
num_cluster=[50]
window_size=[2,3,5,7,11,15]
step_size=[1,2,3,5,7,10]


for K in num_cluster:
    Classic_result=[]
    GMM_result=[]
    IF_result=[]
    LOF_result=[]
    OCSVM_result=[]
    for ss in step_size:
        print("step:",ss)
        for c in range(1):
            kmeans=KMeans(n_clusters=K)
            GMM=GaussianMixture(n_components=1)
            IF=IsolationForest()
            LOF=LocalOutlierFactor(novelty=True)
            OCSVM=OneClassSVM()
            
            print(c)
            train = X_train[y_train==c]
            #result=[]
            full_LCB=np.zeros([len(train),0])
            full_LCB_test=np.zeros([len(X_test),0])
            for size in window_size:
                print(size)
                start=time.time()
                i_size=0
                i_index=[]
                for i in range(28-size+1):
                    if i %ss ==0:
                        i_index.append(i)
                        i_size+=1
                        
                j_size=0
                j_index=[]
                for j in range(28-size+1):
                    if j %ss ==0:
                        j_index.append(j)
                        j_size+=1
                i_index=np.array(i_index)
                j_index=np.array(j_index)
                
                #print(size)
                position=np.array(range(size**2)).reshape(size,size)
                windows=np.zeros([len(train),i_size,j_size,size*size])
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,:,num]=train[:,i_index+i][:,:,j_index+j]                
                vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
                
                kmeans.fit(vector)
                cluster=kmeans.predict(vector)
                cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
                #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
                LCB=[]
                for raw in cluster:
                    balance=np.zeros(K)
                    for value in raw:
                        balance[value]+=1
                    LCB.append(balance)
                LCB=np.array(LCB)
                LCB=LCB/cluster.shape[1]
                full_LCB=np.concatenate([full_LCB,LCB],axis=1)
                end=time.time()
                tim=end-start
                print(tim)
                start=time.time()
                windows_test=np.zeros([len(X_test),i_size,j_size,size*size])
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows_test[:,:,:,num]=X_test[:,i_index+i][:,:,j_index+j]
                vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
                cluster_test=kmeans.predict(vector_test)
                cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
                LCB_test=[]
                for raw in cluster_test:
                    balance=np.zeros(K)
                    for value in raw:
                        balance[value]+=1
                    LCB_test.append(balance)
                LCB_test=np.array(LCB_test)
                LCB_test=LCB_test/cluster_test.shape[1]
                full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
                end=time.time()
                tim=end-start
                print(tim)
            GMM.fit(full_LCB)
            IF.fit(full_LCB)
            LOF.fit(full_LCB)
            OCSVM.fit(full_LCB)
            LP=full_LCB.mean(axis=0)
            Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(full_LCB_test)
            IF_seen=IF.score_samples(full_LCB_test)
            LOF_seen=LOF.score_samples(full_LCB_test)
            OCSVM_seen=OCSVM.score_samples(full_LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
        #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
    print(Classic_result)
    print(GMM_result)
    print(IF_result)
    print(LOF_result)
    print(OCSVM_result)
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

step: 1
0
2
10.267699956893921
1.8767485618591309
3
10.139528751373291
2.0773873329162598
5
22.532737493515015
2.680602788925171
7
25.605084657669067
3.4217441082000732
11
45.67159700393677
4.938997030258179
15
48.05364942550659
5.376240968704224
step: 2
0
2
2.844883441925049
0.6076381206512451
3
2.355766534805298
0.5222568511962891
5
4.4897003173828125
0.7099752426147461
7
8.504382371902466
0.9221961498260498
11
11.289611101150513
1.3386237621307373
15
12.013555526733398
1.5278000831604004
step: 3
0
2
1.0092580318450928
0.2635014057159424
3
1.3859522342681885
0.2803523540496826
5
2.2138500213623047
0.3234591484069824
7
4.053079605102539
0.5488488674163818
11
4.911028623580933
0.637216329574585
15
3.932755947113037
0.797856330871582
step: 5
0
2
0.4245288372039795
0.12614130973815918
3
0.48680877685546875
0.12941813468933105
5
0.8672115802764893
0.12876200675964355
7
1.6826343536376953
0.21478843688964844
11
2.0643157958984375
0.3069744110107422
15
0.8940420150756836
0.310349702835083
s

In [8]:
print(np.array(Classic_result))
print(np.array(OCSVM_result))
print(np.array(LOF_result))
print(np.array(IF_result))
print(np.array(GMM_result))
print(np.array(Classic_result).mean())
print(np.array(OCSVM_result).mean())
print(np.array(LOF_result).mean())
print(np.array(IF_result).mean())
print(np.array(GMM_result).mean())

[88.1 93.5 79.6 80.4 84.2 80.1 79.6 86.6 79.5 83.3 87.7 93.6 79.2 78.9
 83.4 80.1 80.4 87.5 79.2 82.  87.  85.9 78.  77.8 81.1 78.1 81.2 87.5
 79.8 84.  92.1 90.9 80.8 85.3 82.8 79.6 85.7 87.8 83.2 87.  87.5 95.8
 74.  69.7 75.8 66.4 78.1 86.4 72.6 83.6 81.3 93.9 67.  70.4 67.7 65.7
 75.9 71.3 63.2 77.8]
[88.7 94.2 77.7 78.  82.8 77.6 80.9 85.4 79.4 82.9 88.7 94.9 78.2 77.4
 82.6 77.9 80.9 86.5 79.2 82.4 87.8 90.1 77.1 77.2 80.8 76.6 82.1 87.2
 79.8 84.2 92.4 93.6 80.1 85.1 82.5 79.2 86.1 87.8 83.  87.  90.  95.5
 76.9 72.8 78.8 72.6 83.4 88.  75.8 85.8 86.5 94.1 71.6 76.7 73.  70.2
 87.  77.1 72.1 82.7]
[97.3 98.6 96.2 97.4 96.9 97.5 98.8 97.3 96.2 97.7 96.9 99.  96.  95.3
 94.6 96.2 98.3 96.2 94.3 96.2 94.  98.9 88.5 90.7 90.6 92.  97.5 93.6
 88.  94.5 87.9 99.3 82.  84.3 87.8 90.7 92.8 92.5 83.2 92.  83.5 98.1
 80.2 85.7 83.8 88.6 93.4 87.6 83.3 91.5 93.2 81.6 78.2 79.8 89.4 85.5
 92.6 84.5 78.8 89.4]
[91.4 99.4 86.7 91.6 89.7 91.4 86.7 90.4 86.8 88.2 88.8 99.1 84.7 86.6
 86.4 89.1 

In [20]:
pd.DataFrame(np.array(GMM_result).reshape(6,10)).to_csv("GMM_FMNIST-fusion.csv")

In [11]:
X_train=mnist.load_data()[0][0]
y_train=mnist.load_data()[0][1]
X_test=mnist.load_data()[1][0]
y_test=mnist.load_data()[1][1]
num_cluster=[50]
window_size=[2,3,5,7,11,15]
for K in num_cluster:
    Classic_result=[]
    GMM_result=[]
    IF_result=[]
    LOF_result=[]
    OCSVM_result=[]
    for c in range(10):
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        
        print(c)
        train = X_train[y_train==c]
        #result=[]
        full_LCB=np.zeros([len(train),0])
        full_LCB_test=np.zeros([len(X_test),0])
        for size in window_size:
            print(size)
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            LCB=LCB/cluster.shape[1]
            full_LCB=np.concatenate([full_LCB,LCB],axis=1)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            LCB_test=LCB_test/cluster_test.shape[1]
            full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
        GMM.fit(full_LCB)
        IF.fit(full_LCB)
        LOF.fit(full_LCB)
        OCSVM.fit(full_LCB)
        LP=full_LCB.mean(axis=0)
        Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
        GMM_seen=GMM.score_samples(full_LCB_test)
        IF_seen=IF.score_samples(full_LCB_test)
        LOF_seen=LOF.score_samples(full_LCB_test)
        OCSVM_seen=OCSVM.score_samples(full_LCB_test)
        y_eval=np.zeros(len(y_test))
        y_eval[y_test==c]=1
        Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
        GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
        IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
        LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
        OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
        #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)
    print(Classic_result)
    print(OCSVM_result)
    print(LOF_result)
    print(IF_result)
    print(GMM_result)
    print(np.array(Classic_result).mean())
    print(np.array(OCSVM_result).mean())
    print(np.array(LOF_result).mean())
    print(np.array(IF_result).mean())
    print(np.array(GMM_result).mean())
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

0
2
3
5
7
11
15
[89.2]
[88.4]
[97.4]
[91.3]
[99.6]
1
2
3
5
7
11
15
[89.2, 85.4]
[88.4, 94.5]
[97.4, 97.6]
[91.3, 99.7]
[99.6, 99.8]
2
2
3
5
7
11
15
[89.2, 85.4, 83.3]
[88.4, 94.5, 78.0]
[97.4, 97.6, 98.0]
[91.3, 99.7, 89.0]
[99.6, 99.8, 97.1]
3
2
3
5
7
11
15
[89.2, 85.4, 83.3, 88.9]
[88.4, 94.5, 78.0, 82.1]
[97.4, 97.6, 98.0, 98.3]
[91.3, 99.7, 89.0, 95.1]
[99.6, 99.8, 97.1, 98.6]
4
2
3
5
7
11
15
[89.2, 85.4, 83.3, 88.9, 79.2]
[88.4, 94.5, 78.0, 82.1, 78.8]
[97.4, 97.6, 98.0, 98.3, 97.3]
[91.3, 99.7, 89.0, 95.1, 91.7]
[99.6, 99.8, 97.1, 98.6, 97.3]
5
2
3
5
7
11
15
[89.2, 85.4, 83.3, 88.9, 79.2, 88.8]
[88.4, 94.5, 78.0, 82.1, 78.8, 79.0]
[97.4, 97.6, 98.0, 98.3, 97.3, 98.3]
[91.3, 99.7, 89.0, 95.1, 91.7, 96.5]
[99.6, 99.8, 97.1, 98.6, 97.3, 98.6]
6
2
3
5
7
11
15
[89.2, 85.4, 83.3, 88.9, 79.2, 88.8, 84.7]
[88.4, 94.5, 78.0, 82.1, 78.8, 79.0, 81.2]
[97.4, 97.6, 98.0, 98.3, 97.3, 98.3, 99.4]
[91.3, 99.7, 89.0, 95.1, 91.7, 96.5, 90.4]
[99.6, 99.8, 97.1, 98.6, 97.3, 98.6, 99.2]
7
2
3
5
7
11


In [3]:

X_train=mnist.load_data()[0][0]
y_train=mnist.load_data()[0][1]
X_test=mnist.load_data()[1][0]
y_test=mnist.load_data()[1][1]
num_cluster=[25]
window_size=[2,3,5,7,11,15]
for K in num_cluster:
    Classic_result=[]
    GMM_result=[]
    IF_result=[]
    LOF_result=[]
    OCSVM_result=[]
    for c in range(10):
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        
        print(c)
        train = X_train[y_train==c]
        #result=[]
        full_LCB=np.zeros([len(train),0])
        full_LCB_test=np.zeros([len(X_test),0])
        for size in window_size:
            print(size)
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            LCB=LCB/cluster.shape[1]
            full_LCB=np.concatenate([full_LCB,LCB],axis=1)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            LCB_test=LCB_test/cluster_test.shape[1]
            full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
        GMM.fit(full_LCB)
        IF.fit(full_LCB)
        LOF.fit(full_LCB)
        OCSVM.fit(full_LCB)
        LP=full_LCB.mean(axis=0)
        Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
        GMM_seen=GMM.score_samples(full_LCB_test)
        IF_seen=IF.score_samples(full_LCB_test)
        LOF_seen=LOF.score_samples(full_LCB_test)
        OCSVM_seen=OCSVM.score_samples(full_LCB_test)
        y_eval=np.zeros(len(y_test))
        y_eval[y_test==c]=1
        Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
        GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
        IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
        LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
        OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
        #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)
    print(Classic_result)
    print(OCSVM_result)
    print(LOF_result)
    print(IF_result)
    print(GMM_result)
    print(np.array(Classic_result).mean())
    print(np.array(OCSVM_result).mean())
    print(np.array(LOF_result).mean())
    print(np.array(IF_result).mean())
    print(np.array(GMM_result).mean())
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

0
2
3
5
7
11
15
[91.1]
[90.1]
[98.9]
[91.9]
[99.6]
1
2
3
5
7
11
15
[91.1, 82.6]
[90.1, 94.1]
[98.9, 98.1]
[91.9, 99.6]
[99.6, 99.8]
2
2
3
5
7
11
15
[91.1, 82.6, 83.3]
[90.1, 94.1, 78.9]
[98.9, 98.1, 97.9]
[91.9, 99.6, 86.9]
[99.6, 99.8, 96.1]
3
2
3
5
7
11
15
[91.1, 82.6, 83.3, 87.9]
[90.1, 94.1, 78.9, 82.0]
[98.9, 98.1, 97.9, 98.4]
[91.9, 99.6, 86.9, 93.4]
[99.6, 99.8, 96.1, 97.8]
4
2
3
5
7
11
15
[91.1, 82.6, 83.3, 87.9, 80.1]
[90.1, 94.1, 78.9, 82.0, 79.6]
[98.9, 98.1, 97.9, 98.4, 97.1]
[91.9, 99.6, 86.9, 93.4, 92.1]
[99.6, 99.8, 96.1, 97.8, 96.8]
5
2
3
5
7
11
15
[91.1, 82.6, 83.3, 87.9, 80.1, 88.5]
[90.1, 94.1, 78.9, 82.0, 79.6, 81.3]
[98.9, 98.1, 97.9, 98.4, 97.1, 98.4]
[91.9, 99.6, 86.9, 93.4, 92.1, 94.4]
[99.6, 99.8, 96.1, 97.8, 96.8, 98.2]
6
2
3
5
7
11
15
[91.1, 82.6, 83.3, 87.9, 80.1, 88.5, 84.2]
[90.1, 94.1, 78.9, 82.0, 79.6, 81.3, 82.6]
[98.9, 98.1, 97.9, 98.4, 97.1, 98.4, 99.2]
[91.9, 99.6, 86.9, 93.4, 92.1, 94.4, 89.8]
[99.6, 99.8, 96.1, 97.8, 96.8, 98.2, 98.7]
7
2
3
5
7
11


In [13]:
X_train=fmnist.load_data()[0][0]
y_train=fmnist.load_data()[0][1]
X_test=fmnist.load_data()[1][0]
y_test=fmnist.load_data()[1][1]
num_cluster=[50]
window_size=[2,3,5,7,11,15]
for K in num_cluster:
    Classic_result=[]
    GMM_result=[]
    IF_result=[]
    LOF_result=[]
    OCSVM_result=[]
    for c in range(10):
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        
        print(c)
        train = X_train[y_train==c]
        #result=[]
        full_LCB=np.zeros([len(train),0])
        full_LCB_test=np.zeros([len(X_test),0])
        for size in window_size:
            print(size)
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            LCB=LCB/cluster.shape[1]
            full_LCB=np.concatenate([full_LCB,LCB],axis=1)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            LCB_test=LCB_test/cluster_test.shape[1]
            full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
        GMM.fit(full_LCB)
        IF.fit(full_LCB)
        LOF.fit(full_LCB)
        OCSVM.fit(full_LCB)
        LP=full_LCB.mean(axis=0)
        Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
        GMM_seen=GMM.score_samples(full_LCB_test)
        IF_seen=IF.score_samples(full_LCB_test)
        LOF_seen=LOF.score_samples(full_LCB_test)
        OCSVM_seen=OCSVM.score_samples(full_LCB_test)
        y_eval=np.zeros(len(y_test))
        y_eval[y_test==c]=1
        Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
        GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
        IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
        LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
        OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
        #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)
    print(Classic_result)
    print(OCSVM_result)
    print(LOF_result)
    print(IF_result)
    print(GMM_result)
    print(np.array(Classic_result).mean())
    print(np.array(OCSVM_result).mean())
    print(np.array(LOF_result).mean())
    print(np.array(IF_result).mean())
    print(np.array(GMM_result).mean())
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

0
2
3
5
7
11
15
[67.4]
[69.9]
[87.0]
[82.3]
[90.5]
1
2
3
5
7
11
15
[67.4, 95.0]
[69.9, 96.6]
[87.0, 92.4]
[82.3, 93.3]
[90.5, 97.3]
2
2
3
5
7
11
15
[67.4, 95.0, 59.4]
[69.9, 96.6, 71.4]
[87.0, 92.4, 86.9]
[82.3, 93.3, 74.9]
[90.5, 97.3, 85.8]
3
2
3
5
7
11
15
[67.4, 95.0, 59.4, 78.7]
[69.9, 96.6, 71.4, 80.3]
[87.0, 92.4, 86.9, 92.7]
[82.3, 93.3, 74.9, 87.3]
[90.5, 97.3, 85.8, 94.7]
4
2
3
5
7
11
15
[67.4, 95.0, 59.4, 78.7, 75.2]
[69.9, 96.6, 71.4, 80.3, 75.4]
[87.0, 92.4, 86.9, 92.7, 88.2]
[82.3, 93.3, 74.9, 87.3, 85.0]
[90.5, 97.3, 85.8, 94.7, 91.1]
5
2
3
5
7
11
15
[67.4, 95.0, 59.4, 78.7, 75.2, 95.5]
[69.9, 96.6, 71.4, 80.3, 75.4, 93.9]
[87.0, 92.4, 86.9, 92.7, 88.2, 93.9]
[82.3, 93.3, 74.9, 87.3, 85.0, 92.4]
[90.5, 97.3, 85.8, 94.7, 91.1, 93.6]
6
2
3
5
7
11
15
[67.4, 95.0, 59.4, 78.7, 75.2, 95.5, 49.2]
[69.9, 96.6, 71.4, 80.3, 75.4, 93.9, 58.8]
[87.0, 92.4, 86.9, 92.7, 88.2, 93.9, 80.4]
[82.3, 93.3, 74.9, 87.3, 85.0, 92.4, 73.9]
[90.5, 97.3, 85.8, 94.7, 91.1, 93.6, 81.2]
7
2
3
5
7
11


In [14]:
X_train=np.load("k49-train-imgs.npz")["arr_0"]
y_train=np.load("k49-train-labels.npz")["arr_0"]
X_test=np.load("k49-test-imgs.npz")["arr_0"]
y_test=np.load("k49-test-labels.npz")["arr_0"]
num_cluster=[50]
window_size=[2,3,5,7,11,15]
for K in num_cluster:
    Classic_result=[]
    GMM_result=[]
    IF_result=[]
    LOF_result=[]
    OCSVM_result=[]
    for c in range(49):
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        
        print(c)
        train = X_train[y_train==c]
        #result=[]
        full_LCB=np.zeros([len(train),0])
        full_LCB_test=np.zeros([len(X_test),0])
        for size in window_size:
            print(size)
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            LCB=LCB/cluster.shape[1]
            full_LCB=np.concatenate([full_LCB,LCB],axis=1)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            LCB_test=LCB_test/cluster_test.shape[1]
            full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
        GMM.fit(full_LCB)
        IF.fit(full_LCB)
        LOF.fit(full_LCB)
        OCSVM.fit(full_LCB)
        LP=full_LCB.mean(axis=0)
        Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
        GMM_seen=GMM.score_samples(full_LCB_test)
        IF_seen=IF.score_samples(full_LCB_test)
        LOF_seen=LOF.score_samples(full_LCB_test)
        OCSVM_seen=OCSVM.score_samples(full_LCB_test)
        y_eval=np.zeros(len(y_test))
        y_eval[y_test==c]=1
        Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
        GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
        IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
        LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
        OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
        #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)
    print(Classic_result)
    print(OCSVM_result)
    print(LOF_result)
    print(IF_result)
    print(GMM_result)
    print(np.array(Classic_result).mean())
    print(np.array(OCSVM_result).mean())
    print(np.array(LOF_result).mean())
    print(np.array(IF_result).mean())
    print(np.array(GMM_result).mean())
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

0
2
3
5
7
11
15
[82.2]
[81.7]
[93.9]
[84.5]
[85.1]
1
2
3
5
7
11
15
[82.2, 68.8]
[81.7, 68.3]
[93.9, 97.6]
[84.5, 79.7]
[85.1, 94.9]
2
2
3
5
7
11
15
[82.2, 68.8, 94.5]
[81.7, 68.3, 92.4]
[93.9, 97.6, 98.7]
[84.5, 79.7, 97.4]
[85.1, 94.9, 99.0]
3
2
3
5
7
11
15
[82.2, 68.8, 94.5, 63.8]
[81.7, 68.3, 92.4, 71.7]
[93.9, 97.6, 98.7, 96.8]
[84.5, 79.7, 97.4, 90.5]
[85.1, 94.9, 99.0, 96.2]
4
2
3
5
7
11
15
[82.2, 68.8, 94.5, 63.8, 71.7]
[81.7, 68.3, 92.4, 71.7, 69.7]
[93.9, 97.6, 98.7, 96.8, 94.6]
[84.5, 79.7, 97.4, 90.5, 78.0]
[85.1, 94.9, 99.0, 96.2, 92.0]
5
2
3
5
7
11
15
[82.2, 68.8, 94.5, 63.8, 71.7, 73.2]
[81.7, 68.3, 92.4, 71.7, 69.7, 71.5]
[93.9, 97.6, 98.7, 96.8, 94.6, 90.4]
[84.5, 79.7, 97.4, 90.5, 78.0, 81.8]
[85.1, 94.9, 99.0, 96.2, 92.0, 89.1]
6
2
3
5
7
11
15
[82.2, 68.8, 94.5, 63.8, 71.7, 73.2, 68.9]
[81.7, 68.3, 92.4, 71.7, 69.7, 71.5, 62.6]
[93.9, 97.6, 98.7, 96.8, 94.6, 90.4, 93.4]
[84.5, 79.7, 97.4, 90.5, 78.0, 81.8, 81.1]
[85.1, 94.9, 99.0, 96.2, 92.0, 89.1, 85.3]
7
2
3
5
7
11


3
5
7
11
15
[82.2, 68.8, 94.5, 63.8, 71.7, 73.2, 68.9, 69.8, 64.0, 68.4, 79.7, 64.4, 70.2, 79.3, 75.5, 69.6, 92.1, 48.6, 81.6, 71.4, 71.1, 68.8, 86.0, 76.9]
[81.7, 68.3, 92.4, 71.7, 69.7, 71.5, 62.6, 72.1, 66.9, 66.9, 77.4, 65.2, 66.1, 77.8, 72.4, 65.5, 86.5, 60.9, 80.4, 66.2, 74.7, 68.1, 85.2, 78.8]
[93.9, 97.6, 98.7, 96.8, 94.6, 90.4, 93.4, 95.6, 88.6, 91.1, 92.3, 92.2, 87.3, 78.3, 87.9, 93.6, 96.8, 95.6, 95.3, 92.9, 84.6, 92.2, 90.5, 91.0]
[84.5, 79.7, 97.4, 90.5, 78.0, 81.8, 81.1, 88.8, 67.7, 75.7, 81.9, 83.3, 75.7, 77.4, 79.4, 76.2, 96.6, 66.9, 90.6, 82.1, 70.0, 71.1, 84.7, 65.8]
[85.1, 94.9, 99.0, 96.2, 92.0, 89.1, 85.3, 95.3, 76.9, 86.4, 87.6, 89.5, 82.1, 81.4, 87.3, 84.9, 98.4, 83.6, 94.8, 90.0, 74.7, 80.9, 90.8, 81.3]
24
2
3
5
7
11
15
[82.2, 68.8, 94.5, 63.8, 71.7, 73.2, 68.9, 69.8, 64.0, 68.4, 79.7, 64.4, 70.2, 79.3, 75.5, 69.6, 92.1, 48.6, 81.6, 71.4, 71.1, 68.8, 86.0, 76.9, 67.8]
[81.7, 68.3, 92.4, 71.7, 69.7, 71.5, 62.6, 72.1, 66.9, 66.9, 77.4, 65.2, 66.1, 77.8, 72.4, 65.5

3
5
7
11
15
[82.2, 68.8, 94.5, 63.8, 71.7, 73.2, 68.9, 69.8, 64.0, 68.4, 79.7, 64.4, 70.2, 79.3, 75.5, 69.6, 92.1, 48.6, 81.6, 71.4, 71.1, 68.8, 86.0, 76.9, 67.8, 79.0, 84.5, 68.2, 83.0, 59.8, 67.8, 58.5, 78.7, 83.6]
[81.7, 68.3, 92.4, 71.7, 69.7, 71.5, 62.6, 72.1, 66.9, 66.9, 77.4, 65.2, 66.1, 77.8, 72.4, 65.5, 86.5, 60.9, 80.4, 66.2, 74.7, 68.1, 85.2, 78.8, 70.3, 77.7, 84.5, 66.7, 81.5, 65.5, 65.7, 60.7, 77.9, 83.4]
[93.9, 97.6, 98.7, 96.8, 94.6, 90.4, 93.4, 95.6, 88.6, 91.1, 92.3, 92.2, 87.3, 78.3, 87.9, 93.6, 96.8, 95.6, 95.3, 92.9, 84.6, 92.2, 90.5, 91.0, 87.4, 90.4, 94.9, 91.7, 92.4, 95.5, 90.0, 91.9, 94.0, 94.9]
[84.5, 79.7, 97.4, 90.5, 78.0, 81.8, 81.1, 88.8, 67.7, 75.7, 81.9, 83.3, 75.7, 77.4, 79.4, 76.2, 96.6, 66.9, 90.6, 82.1, 70.0, 71.1, 84.7, 65.8, 79.2, 82.9, 84.7, 74.3, 85.3, 62.3, 71.7, 74.7, 84.7, 88.9]
[85.1, 94.9, 99.0, 96.2, 92.0, 89.1, 85.3, 95.3, 76.9, 86.4, 87.6, 89.5, 82.1, 81.4, 87.3, 84.9, 98.4, 83.6, 94.8, 90.0, 74.7, 80.9, 90.8, 81.3, 82.1, 88.2, 86.6, 81.6,

3
5
7
11
15
[82.2, 68.8, 94.5, 63.8, 71.7, 73.2, 68.9, 69.8, 64.0, 68.4, 79.7, 64.4, 70.2, 79.3, 75.5, 69.6, 92.1, 48.6, 81.6, 71.4, 71.1, 68.8, 86.0, 76.9, 67.8, 79.0, 84.5, 68.2, 83.0, 59.8, 67.8, 58.5, 78.7, 83.6, 76.9, 70.0, 73.1, 88.9, 85.8, 80.5, 66.3, 71.7]
[81.7, 68.3, 92.4, 71.7, 69.7, 71.5, 62.6, 72.1, 66.9, 66.9, 77.4, 65.2, 66.1, 77.8, 72.4, 65.5, 86.5, 60.9, 80.4, 66.2, 74.7, 68.1, 85.2, 78.8, 70.3, 77.7, 84.5, 66.7, 81.5, 65.5, 65.7, 60.7, 77.9, 83.4, 75.1, 68.9, 78.6, 87.0, 81.4, 78.6, 63.2, 71.7]
[93.9, 97.6, 98.7, 96.8, 94.6, 90.4, 93.4, 95.6, 88.6, 91.1, 92.3, 92.2, 87.3, 78.3, 87.9, 93.6, 96.8, 95.6, 95.3, 92.9, 84.6, 92.2, 90.5, 91.0, 87.4, 90.4, 94.9, 91.7, 92.4, 95.5, 90.0, 91.9, 94.0, 94.9, 91.7, 90.9, 95.7, 97.5, 96.3, 92.7, 85.5, 89.8]
[84.5, 79.7, 97.4, 90.5, 78.0, 81.8, 81.1, 88.8, 67.7, 75.7, 81.9, 83.3, 75.7, 77.4, 79.4, 76.2, 96.6, 66.9, 90.6, 82.1, 70.0, 71.1, 84.7, 65.8, 79.2, 82.9, 84.7, 74.3, 85.3, 62.3, 71.7, 74.7, 84.7, 88.9, 77.2, 61.9, 75.0, 93.3, 

3
5
7
11
15
[82.2, 68.8, 94.5, 63.8, 71.7, 73.2, 68.9, 69.8, 64.0, 68.4, 79.7, 64.4, 70.2, 79.3, 75.5, 69.6, 92.1, 48.6, 81.6, 71.4, 71.1, 68.8, 86.0, 76.9, 67.8, 79.0, 84.5, 68.2, 83.0, 59.8, 67.8, 58.5, 78.7, 83.6, 76.9, 70.0, 73.1, 88.9, 85.8, 80.5, 66.3, 71.7, 76.5, 69.8, 82.1, 92.7, 72.2, 89.6, 66.6]
[81.7, 68.3, 92.4, 71.7, 69.7, 71.5, 62.6, 72.1, 66.9, 66.9, 77.4, 65.2, 66.1, 77.8, 72.4, 65.5, 86.5, 60.9, 80.4, 66.2, 74.7, 68.1, 85.2, 78.8, 70.3, 77.7, 84.5, 66.7, 81.5, 65.5, 65.7, 60.7, 77.9, 83.4, 75.1, 68.9, 78.6, 87.0, 81.4, 78.6, 63.2, 71.7, 77.7, 67.1, 80.1, 90.4, 73.8, 88.9, 66.9]
[93.9, 97.6, 98.7, 96.8, 94.6, 90.4, 93.4, 95.6, 88.6, 91.1, 92.3, 92.2, 87.3, 78.3, 87.9, 93.6, 96.8, 95.6, 95.3, 92.9, 84.6, 92.2, 90.5, 91.0, 87.4, 90.4, 94.9, 91.7, 92.4, 95.5, 90.0, 91.9, 94.0, 94.9, 91.7, 90.9, 95.7, 97.5, 96.3, 92.7, 85.5, 89.8, 95.9, 94.7, 86.4, 93.2, 91.1, 97.6, 94.8]
[84.5, 79.7, 97.4, 90.5, 78.0, 81.8, 81.1, 88.8, 67.7, 75.7, 81.9, 83.3, 75.7, 77.4, 79.4, 76.2, 96.6, 

In [16]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[50]
window_size=[2,3,5,7,11,15]
for K in num_cluster:
    Classic_result=[]
    GMM_result=[]
    IF_result=[]
    LOF_result=[]
    OCSVM_result=[]
    for c in range(10):
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        
        print(c)
        train = X_train[y_train==c]
        #result=[]
        full_LCB=np.zeros([len(train),0])
        full_LCB_test=np.zeros([len(X_test),0])
        for size in window_size:
            print(size)
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num*3:num*3+3]=train[:,i:32-size+1+i,j:32-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            LCB=LCB/cluster.shape[1]
            full_LCB=np.concatenate([full_LCB,LCB],axis=1)
            windows_test=np.zeros([len(X_test),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num*3:num*3+3]=X_test[:,i:32-size+1+i,j:32-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            LCB_test=LCB_test/cluster_test.shape[1]
            full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
        GMM.fit(full_LCB)
        IF.fit(full_LCB)
        LOF.fit(full_LCB)
        OCSVM.fit(full_LCB)
        LP=full_LCB.mean(axis=0)
        Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
        GMM_seen=GMM.score_samples(full_LCB_test)
        IF_seen=IF.score_samples(full_LCB_test)
        LOF_seen=LOF.score_samples(full_LCB_test)
        OCSVM_seen=OCSVM.score_samples(full_LCB_test)
        y_eval=np.zeros(len(y_test))
        y_eval[y_test==c]=1
        Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
        GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
        IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
        LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
        OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
        #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)
    print(Classic_result)
    print(OCSVM_result)
    print(LOF_result)
    print(IF_result)
    print(GMM_result)
    print(np.array(Classic_result).mean())
    print(np.array(OCSVM_result).mean())
    print(np.array(LOF_result).mean())
    print(np.array(IF_result).mean())
    print(np.array(GMM_result).mean())
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

0
2
3
5
7
11
15
[44.9]
[41.8]
[48.6]
[62.0]
[57.7]
1
2
3
5
7
11
15
[44.9, 74.7]
[41.8, 75.8]
[48.6, 83.0]
[62.0, 70.3]
[57.7, 76.3]
2
2
3
5
7
11
15
[44.9, 74.7, 41.3]
[41.8, 75.8, 37.8]
[48.6, 83.0, 53.1]
[62.0, 70.3, 62.1]
[57.7, 76.3, 56.0]
3
2
3
5
7
11
15
[44.9, 74.7, 41.3, 59.3]
[41.8, 75.8, 37.8, 59.9]
[48.6, 83.0, 53.1, 65.9]
[62.0, 70.3, 62.1, 53.7]
[57.7, 76.3, 56.0, 66.8]
4
2
3
5
7
11
15
[44.9, 74.7, 41.3, 59.3, 48.4]
[41.8, 75.8, 37.8, 59.9, 44.8]
[48.6, 83.0, 53.1, 65.9, 52.0]
[62.0, 70.3, 62.1, 53.7, 71.0]
[57.7, 76.3, 56.0, 66.8, 65.4]
5
2
3
5
7
11
15
[44.9, 74.7, 41.3, 59.3, 48.4, 62.1]
[41.8, 75.8, 37.8, 59.9, 44.8, 62.4]
[48.6, 83.0, 53.1, 65.9, 52.0, 68.5]
[62.0, 70.3, 62.1, 53.7, 71.0, 56.4]
[57.7, 76.3, 56.0, 66.8, 65.4, 67.0]
6
2
3
5
7
11
15
[44.9, 74.7, 41.3, 59.3, 48.4, 62.1, 57.3]
[41.8, 75.8, 37.8, 59.9, 44.8, 62.4, 53.5]
[48.6, 83.0, 53.1, 65.9, 52.0, 68.5, 66.5]
[62.0, 70.3, 62.1, 53.7, 71.0, 56.4, 74.5]
[57.7, 76.3, 56.0, 66.8, 65.4, 67.0, 76.9]
7
2
3
5
7
11


In [17]:
from tensorflow.keras.datasets import cifar100

In [18]:
X_train=cifar100.load_data(label_mode="coarse")[0][0]
y_train=cifar100.load_data(label_mode="coarse")[0][1]
X_test=cifar100.load_data(label_mode="coarse")[1][0]
y_test=cifar100.load_data(label_mode="coarse")[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[50]
window_size=[2,3,5,7,11,15]
for K in num_cluster:
    Classic_result=[]
    GMM_result=[]
    IF_result=[]
    LOF_result=[]
    OCSVM_result=[]
    for c in range(20):
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        
        print(c)
        train = X_train[y_train==c]
        #result=[]
        full_LCB=np.zeros([len(train),0])
        full_LCB_test=np.zeros([len(X_test),0])
        for size in window_size:
            print(size)
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num*3:num*3+3]=train[:,i:32-size+1+i,j:32-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            LCB=LCB/cluster.shape[1]
            full_LCB=np.concatenate([full_LCB,LCB],axis=1)
            windows_test=np.zeros([len(X_test),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num*3:num*3+3]=X_test[:,i:32-size+1+i,j:32-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            LCB_test=LCB_test/cluster_test.shape[1]
            full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
        GMM.fit(full_LCB)
        IF.fit(full_LCB)
        LOF.fit(full_LCB)
        OCSVM.fit(full_LCB)
        LP=full_LCB.mean(axis=0)
        Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
        GMM_seen=GMM.score_samples(full_LCB_test)
        IF_seen=IF.score_samples(full_LCB_test)
        LOF_seen=LOF.score_samples(full_LCB_test)
        OCSVM_seen=OCSVM.score_samples(full_LCB_test)
        y_eval=np.zeros(len(y_test))
        y_eval[y_test==c]=1
        Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
        GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
        IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
        LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
        OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
        #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)
    print(Classic_result)
    print(OCSVM_result)
    print(LOF_result)
    print(IF_result)
    print(GMM_result)
    print(np.array(Classic_result).mean())
    print(np.array(OCSVM_result).mean())
    print(np.array(LOF_result).mean())
    print(np.array(IF_result).mean())
    print(np.array(GMM_result).mean())
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

0
2
3
5
7
11
15
[49.7]
[49.1]
[59.6]
[58.1]
[60.3]
1
2
3
5
7
11
15
[49.7, 45.6]
[49.1, 46.7]
[59.6, 57.9]
[58.1, 59.8]
[60.3, 65.5]
2
2
3
5
7
11
15
[49.7, 45.6, 60.9]
[49.1, 46.7, 60.7]
[59.6, 57.9, 67.0]
[58.1, 59.8, 55.2]
[60.3, 65.5, 68.0]
3
2
3
5
7
11
15
[49.7, 45.6, 60.9, 53.4]
[49.1, 46.7, 60.7, 52.5]
[59.6, 57.9, 67.0, 56.7]
[58.1, 59.8, 55.2, 50.6]
[60.3, 65.5, 68.0, 56.9]
4
2
3
5
7
11
15
[49.7, 45.6, 60.9, 53.4, 57.4]
[49.1, 46.7, 60.7, 52.5, 57.6]
[59.6, 57.9, 67.0, 56.7, 65.8]
[58.1, 59.8, 55.2, 50.6, 50.5]
[60.3, 65.5, 68.0, 56.9, 62.1]
5
2
3
5
7
11
15
[49.7, 45.6, 60.9, 53.4, 57.4, 52.5]
[49.1, 46.7, 60.7, 52.5, 57.6, 53.8]
[59.6, 57.9, 67.0, 56.7, 65.8, 55.5]
[58.1, 59.8, 55.2, 50.6, 50.5, 40.0]
[60.3, 65.5, 68.0, 56.9, 62.1, 42.9]
6
2
3
5
7
11
15
[49.7, 45.6, 60.9, 53.4, 57.4, 52.5, 52.4]
[49.1, 46.7, 60.7, 52.5, 57.6, 53.8, 55.3]
[59.6, 57.9, 67.0, 56.7, 65.8, 55.5, 59.5]
[58.1, 59.8, 55.2, 50.6, 50.5, 40.0, 46.6]
[60.3, 65.5, 68.0, 56.9, 62.1, 42.9, 50.5]
7
2
3
5
7
11


In [24]:
X_train=readfile("Xray_pneumonia/train_normal32.pkl")
n=0
for word in X_train:
    if word.shape==(32,32,3):
        X_train[n]=word[:,:,0]
    n+=1
X_train_normal=np.array(X_train)
X_train=readfile("Xray_pneumonia/train_pneumonia32.pkl")
n=0
for word in X_train:
    if word.shape==(32,32,3):
        X_train[n]=word[:,:,0]
    n+=1
X_train_abnormal=np.array(X_train)
X_train=np.concatenate([X_train_normal,X_train_abnormal],axis=0)
y_train=np.zeros(len(X_train))
y_train[len(X_train_normal):]=1
X_test_normal=np.array(readfile("Xray_pneumonia/test_normal32.pkl"))
X_test_abnormal=np.array(readfile("Xray_pneumonia/test_pneumonia32.pkl"))
X_test=np.concatenate([X_test_normal,X_test_abnormal],axis=0)

y_test=np.zeros(len(X_test))
y_test[len(X_test_normal):]=1
num_cluster=[50]
window_size=[2,3,5,7,11,15]

for K in num_cluster:
    Classic_result=[]
    GMM_result=[]
    IF_result=[]
    LOF_result=[]
    OCSVM_result=[]
    for n in range(5):
        for c in range(2):
            kmeans=KMeans(n_clusters=K)
            GMM=GaussianMixture(n_components=1)
            IF=IsolationForest()
            LOF=LocalOutlierFactor(novelty=True)
            OCSVM=OneClassSVM()

            print(c)
            train = X_train[y_train==c]
            #result=[]
            full_LCB=np.zeros([len(train),0])
            full_LCB_test=np.zeros([len(X_test),0])
            for size in window_size:
                print(size)
                position=np.array(range(size**2)).reshape(size,size)
                windows=np.zeros([len(train),32-size+1,32-size+1,size*size])
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,:,num]=train[:,i:32-size+1+i,j:32-size+1+j]
                vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
                kmeans.fit(vector)
                cluster=kmeans.predict(vector)
                cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
                #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
                LCB=[]
                for raw in cluster:
                    balance=np.zeros(K)
                    for value in raw:
                        balance[value]+=1
                    LCB.append(balance)
                LCB=np.array(LCB)
                LCB=LCB/cluster.shape[1]
                full_LCB=np.concatenate([full_LCB,LCB],axis=1)
                windows_test=np.zeros([len(X_test),32-size+1,32-size+1,size*size])
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows_test[:,:,:,num]=X_test[:,i:32-size+1+i,j:32-size+1+j]
                vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
                cluster_test=kmeans.predict(vector_test)
                cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
                LCB_test=[]
                for raw in cluster_test:
                    balance=np.zeros(K)
                    for value in raw:
                        balance[value]+=1
                    LCB_test.append(balance)
                LCB_test=np.array(LCB_test)
                LCB_test=LCB_test/cluster_test.shape[1]
                full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
            GMM.fit(full_LCB)
            IF.fit(full_LCB)
            LOF.fit(full_LCB)
            OCSVM.fit(full_LCB)
            LP=full_LCB.mean(axis=0)
            Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(full_LCB_test)
            IF_seen=IF.score_samples(full_LCB_test)
            LOF_seen=LOF.score_samples(full_LCB_test)
            OCSVM_seen=OCSVM.score_samples(full_LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
        #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)
    print(Classic_result)
    print(OCSVM_result)
    print(LOF_result)
    print(IF_result)
    print(GMM_result)
    print(np.array(Classic_result).mean())
    print(np.array(OCSVM_result).mean())
    print(np.array(LOF_result).mean())
    print(np.array(IF_result).mean())
    print(np.array(GMM_result).mean())
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

0
2
3
5
7
11
15
1
2
3
5
7
11
15
[73.8, 54.3]
[77.3, 55.8]
[79.0, 65.3]
[74.9, 70.0]
[81.9, 67.3]
0
2
3
5
7
11
15
1
2
3
5
7
11
15
[73.8, 54.3, 73.8, 53.9]
[77.3, 55.8, 77.5, 55.4]
[79.0, 65.3, 78.8, 63.4]
[74.9, 70.0, 74.3, 68.0]
[81.9, 67.3, 81.8, 66.2]
0
2
3
5
7
11
15
1
2
3
5
7
11
15
[73.8, 54.3, 73.8, 53.9, 74.2, 53.8]
[77.3, 55.8, 77.5, 55.4, 77.7, 55.7]
[79.0, 65.3, 78.8, 63.4, 78.7, 64.8]
[74.9, 70.0, 74.3, 68.0, 78.3, 64.7]
[81.9, 67.3, 81.8, 66.2, 81.7, 66.7]
0
2
3
5
7
11
15
1
2
3
5
7
11
15
[73.8, 54.3, 73.8, 53.9, 74.2, 53.8, 73.6, 53.4]
[77.3, 55.8, 77.5, 55.4, 77.7, 55.7, 77.4, 55.6]
[79.0, 65.3, 78.8, 63.4, 78.7, 64.8, 79.0, 64.3]
[74.9, 70.0, 74.3, 68.0, 78.3, 64.7, 75.1, 63.1]
[81.9, 67.3, 81.8, 66.2, 81.7, 66.7, 81.5, 66.0]
0
2
3
5
7
11
15
1
2
3
5
7
11
15
[73.8, 54.3, 73.8, 53.9, 74.2, 53.8, 73.6, 53.4, 73.9, 54.7]
[77.3, 55.8, 77.5, 55.4, 77.7, 55.7, 77.4, 55.6, 77.4, 56.3]
[79.0, 65.3, 78.8, 63.4, 78.7, 64.8, 79.0, 64.3, 78.9, 64.6]
[74.9, 70.0, 74.3, 68.0, 78.3, 64.7, 

In [30]:
X_train=readfile("RR-MNIST/X_train.pkl")
X_test=readfile("RR-MNIST/X_test.pkl")
y_train=mnist.load_data()[0][1]
y_test=mnist.load_data()[1][1]
num_cluster=[50]
window_size=[2,3,5,7,11]
for K in num_cluster:
    Classic_result=[]
    GMM_result=[]
    IF_result=[]
    LOF_result=[]
    OCSVM_result=[]
    for c in range(10):
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        
        print(c)
        train = np.array(X_train)[y_train==c]
        #result=[]
        full_LCB=np.zeros([len(train),0])
        full_LCB_test=np.zeros([len(X_test),0])
        for size in window_size:
            print(size)
            position=np.array(range(size**2)).reshape(size,size)
            
            all_windows=np.zeros([0,size*size])
            for n in range(len(train)):
                im_size=train[n].shape[0]
                windows=np.zeros([im_size-size+1,im_size-size+1,size*size])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num]=train[n][i:im_size-size+1+i,j:im_size-size+1+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                all_windows=np.concatenate([all_windows,windows],axis=0)
            kmeans.fit(all_windows)
            
            
            #vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            #kmeans.fit(vector)
            #cluster=kmeans.predict(vector)
            #cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for n in range(len(train)):
                im_size=train[n].shape[0]
                windows=np.zeros([im_size-size+1,im_size-size+1,size*size])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num]=train[n][i:im_size-size+1+i,j:im_size-size+1+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                cluster=kmeans.predict(windows)
                CB=np.zeros(K)
                for k in range(K):
                    CB[k]=len(cluster[cluster==k])
                CB=CB/len(cluster)
                LCB.append(CB)
            LCB=np.array(LCB)
            #LCB=LCB/cluster.shape[1]
            full_LCB=np.concatenate([full_LCB,LCB],axis=1)
            
            LCB_test=[]
            for n in range(len(X_test)):
                im_size=X_test[n].shape[0]
                windows=np.zeros([im_size-size+1,im_size-size+1,size*size])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num]=X_test[n][i:im_size-size+1+i,j:im_size-size+1+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                cluster=kmeans.predict(windows)
                CB=np.zeros(K)
                for k in range(K):
                    CB[k]=len(cluster[cluster==k])
                CB=CB/len(cluster)
                LCB_test.append(CB)
            LCB_test=np.array(LCB_test)
            #LCB_test=LCB_test/cluster_test.shape[1]
            full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
        GMM.fit(full_LCB)
        IF.fit(full_LCB)
        LOF.fit(full_LCB)
        OCSVM.fit(full_LCB)
        LP=full_LCB.mean(axis=0)
        Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
        GMM_seen=GMM.score_samples(full_LCB_test)
        IF_seen=IF.score_samples(full_LCB_test)
        LOF_seen=LOF.score_samples(full_LCB_test)
        OCSVM_seen=OCSVM.score_samples(full_LCB_test)
        y_eval=np.zeros(len(y_test))
        y_eval[y_test==c]=1
        Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
        GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
        IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
        LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
        OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
        #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)
    print(Classic_result)
    print(OCSVM_result)
    print(LOF_result)
    print(IF_result)
    print(GMM_result)
    print(np.array(Classic_result).mean())
    print(np.array(OCSVM_result).mean())
    print(np.array(LOF_result).mean())
    print(np.array(IF_result).mean())
    print(np.array(GMM_result).mean())
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

0
2
3
5
7
11
[63.8]
[68.5]
[94.5]
[56.7]
[61.1]
1
2
3
5
7
11
[63.8, 67.6]
[68.5, 55.6]
[94.5, 93.6]
[56.7, 92.9]
[61.1, 90.3]
2
2
3
5
7
11
[63.8, 67.6, 63.5]
[68.5, 55.6, 63.0]
[94.5, 93.6, 87.5]
[56.7, 92.9, 63.1]
[61.1, 90.3, 59.5]
3
2
3
5
7
11
[63.8, 67.6, 63.5, 64.7]
[68.5, 55.6, 63.0, 59.2]
[94.5, 93.6, 87.5, 95.0]
[56.7, 92.9, 63.1, 67.9]
[61.1, 90.3, 59.5, 66.8]
4
2
3
5
7
11
[63.8, 67.6, 63.5, 64.7, 60.6]
[68.5, 55.6, 63.0, 59.2, 55.2]
[94.5, 93.6, 87.5, 95.0, 93.2]
[56.7, 92.9, 63.1, 67.9, 66.0]
[61.1, 90.3, 59.5, 66.8, 67.4]
5
2
3
5
7
11
[63.8, 67.6, 63.5, 64.7, 60.6, 64.0]
[68.5, 55.6, 63.0, 59.2, 55.2, 58.6]
[94.5, 93.6, 87.5, 95.0, 93.2, 92.7]
[56.7, 92.9, 63.1, 67.9, 66.0, 71.5]
[61.1, 90.3, 59.5, 66.8, 67.4, 65.9]
6
2
3
5
7
11
[63.8, 67.6, 63.5, 64.7, 60.6, 64.0, 61.1]
[68.5, 55.6, 63.0, 59.2, 55.2, 58.6, 56.9]
[94.5, 93.6, 87.5, 95.0, 93.2, 92.7, 91.8]
[56.7, 92.9, 63.1, 67.9, 66.0, 71.5, 64.0]
[61.1, 90.3, 59.5, 66.8, 67.4, 65.9, 63.5]
7
2
3
5
7
11
[63.8, 67.6, 63.5, 64

In [ ]:

            
position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),ima-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num*3:num*3+3]=train[:,i:imsize-size+1+i,j:imsize-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])            

In [32]:
X_train=readfile("RR-CIFAR10/X_train.pkl")
X_test=readfile("RR-CIFAR10/X_test.pkl")
y_train=cifar10.load_data()[0][1]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[50]
window_size=[2,3,5,7,11]
for K in num_cluster:
    Classic_result=[]
    GMM_result=[]
    IF_result=[]
    LOF_result=[]
    OCSVM_result=[]
    for c in range(10):
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        
        print(c)
        train = np.array(X_train)[y_train==c]
        #result=[]
        full_LCB=np.zeros([len(train),0])
        full_LCB_test=np.zeros([len(X_test),0])
        for size in window_size:
            print(size)
            position=np.array(range(size**2)).reshape(size,size)
            
            all_windows=np.zeros([0,size*size*3])
            for n in range(len(train)):
                im_size=train[n].shape[0]
                windows=np.zeros([im_size-size+1,im_size-size+1,size*size*3])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num*3:num*3+3]=train[n][i:im_size-size+1+i,j:im_size-size+1+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                all_windows=np.concatenate([all_windows,windows],axis=0)
            kmeans.fit(all_windows)
            
            
            #vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            #kmeans.fit(vector)
            #cluster=kmeans.predict(vector)
            #cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for n in range(len(train)):
                im_size=train[n].shape[0]
                windows=np.zeros([im_size-size+1,im_size-size+1,size*size*3])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num*3:num*3+3]=train[n][i:im_size-size+1+i,j:im_size-size+1+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                cluster=kmeans.predict(windows)
                CB=np.zeros(K)
                for k in range(K):
                    CB[k]=len(cluster[cluster==k])
                CB=CB/len(cluster)
                LCB.append(CB)
            LCB=np.array(LCB)
            #LCB=LCB/cluster.shape[1]
            full_LCB=np.concatenate([full_LCB,LCB],axis=1)
            
            LCB_test=[]
            for n in range(len(X_test)):
                im_size=X_test[n].shape[0]
                windows=np.zeros([im_size-size+1,im_size-size+1,size*size*3])
                position=np.array(range(size**2)).reshape(size,size)
                for i in range(size):
                    for j in range(size):
                        num=position[i][j]
                        windows[:,:,num*3:num*3+3]=X_test[n][i:im_size-size+1+i,j:im_size-size+1+j]
                windows=windows.reshape(windows.shape[0]*windows.shape[1],windows.shape[2])
                cluster=kmeans.predict(windows)
                CB=np.zeros(K)
                for k in range(K):
                    CB[k]=len(cluster[cluster==k])
                CB=CB/len(cluster)
                LCB_test.append(CB)
            LCB_test=np.array(LCB_test)
            #LCB_test=LCB_test/cluster_test.shape[1]
            full_LCB_test=np.concatenate([full_LCB_test,LCB_test],axis=1)
        GMM.fit(full_LCB)
        IF.fit(full_LCB)
        LOF.fit(full_LCB)
        OCSVM.fit(full_LCB)
        LP=full_LCB.mean(axis=0)
        Unseen_score=(abs(full_LCB_test-LP)).sum(axis=1)
        GMM_seen=GMM.score_samples(full_LCB_test)
        IF_seen=IF.score_samples(full_LCB_test)
        LOF_seen=LOF.score_samples(full_LCB_test)
        OCSVM_seen=OCSVM.score_samples(full_LCB_test)
        y_eval=np.zeros(len(y_test))
        y_eval[y_test==c]=1
        Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
        GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
        IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
        LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
        OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
        #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        print(Classic_result)
        print(OCSVM_result)
        print(LOF_result)
        print(IF_result)
        print(GMM_result)
    print(Classic_result)
    print(OCSVM_result)
    print(LOF_result)
    print(IF_result)
    print(GMM_result)
    print(np.array(Classic_result).mean())
    print(np.array(OCSVM_result).mean())
    print(np.array(LOF_result).mean())
    print(np.array(IF_result).mean())
    print(np.array(GMM_result).mean())
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

0
2
3
5
7
11
[44.9]
[39.5]
[44.3]
[63.2]
[48.5]
1
2
3
5
7
11
[44.9, 71.3]
[39.5, 72.2]
[44.3, 74.2]
[63.2, 67.6]
[48.5, 65.6]
2
2
3
5
7
11
[44.9, 71.3, 42.2]
[39.5, 72.2, 37.8]
[44.3, 74.2, 51.8]
[63.2, 67.6, 58.3]
[48.5, 65.6, 46.6]
3
2
3
5
7
11
[44.9, 71.3, 42.2, 57.3]
[39.5, 72.2, 37.8, 58.2]
[44.3, 74.2, 51.8, 60.2]
[63.2, 67.6, 58.3, 57.2]
[48.5, 65.6, 46.6, 58.7]
4
2
3
5
7
11
[44.9, 71.3, 42.2, 57.3, 48.8]
[39.5, 72.2, 37.8, 58.2, 46.0]
[44.3, 74.2, 51.8, 60.2, 54.1]
[63.2, 67.6, 58.3, 57.2, 67.1]
[48.5, 65.6, 46.6, 58.7, 56.7]
5
2
3
5
7
11
[44.9, 71.3, 42.2, 57.3, 48.8, 60.0]
[39.5, 72.2, 37.8, 58.2, 46.0, 60.2]
[44.3, 74.2, 51.8, 60.2, 54.1, 61.9]
[63.2, 67.6, 58.3, 57.2, 67.1, 51.5]
[48.5, 65.6, 46.6, 58.7, 56.7, 59.3]
6
2
3
5
7
11
[44.9, 71.3, 42.2, 57.3, 48.8, 60.0, 57.1]
[39.5, 72.2, 37.8, 58.2, 46.0, 60.2, 54.7]
[44.3, 74.2, 51.8, 60.2, 54.1, 61.9, 60.4]
[63.2, 67.6, 58.3, 57.2, 67.1, 51.5, 68.5]
[48.5, 65.6, 46.6, 58.7, 56.7, 59.3, 62.5]
7
2
3
5
7
11
[44.9, 71.3, 42.2, 57